# Train Colosseum on your own dataset

Run this notebook in [Google Colab](https://colab.research.google.com/github/Aqshalikhsan/Colosseum-Microcontroller/blob/main/notebooks/Train_Colosseum.ipynb). Select a GPU runtime first. The notebook reads your dataset from Google Drive and writes checkpoints and metrics back to Drive. Colosseum trains one target class per checkpoint from scratch; the included UC3M weights are not used here.


## 1. Install the framework

The PyPI distribution is `colosseum-microcontroller`; the Python import is `colosseum`.


In [ ]:
%pip install -q --upgrade colosseum-microcontroller
import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())


## 2. Connect Google Drive and choose your dataset

Put your dataset in a folder in **My Drive**. Edit only \`DATASET_DIR\` below to point to that folder. The expected layout is:

\`\`\`text
MyDrive/datasets/my_dataset/
  data.yaml
  images/train/...jpg
  images/val/...jpg
  labels/train/...txt
  labels/val/...txt
\`\`\`

Each label line is \`class_id center_x center_y width height\`, with coordinates from 0 to 1. An image without objects can have an empty or missing label file. If \`data.yaml\` is absent, the cell creates one for this standard folder layout. A manual \`data.yaml\` can use:

\`\`\`yaml
path: .
train: images/train
val: images/val
names: [object]
\`\`\`

The mount prompt asks you to authorize access to your Google Drive. Set \`OUTPUT_DIR\` if you want the results in a different Drive folder.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
import yaml

DRIVE_ROOT = Path('/content/drive/MyDrive')
DATASET_DIR = DRIVE_ROOT / 'datasets' / 'my_dataset'  # Change to your folder in My Drive.
OUTPUT_DIR = DRIVE_ROOT / 'Colosseum'                 # All training results go here.

DATASET_DIR = DATASET_DIR.resolve()
OUTPUT_DIR = OUTPUT_DIR.resolve()
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f'Dataset folder not found: {DATASET_DIR}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_YAML = DATASET_DIR / 'data.yaml'
if not DATA_YAML.is_file():
    required = ('images/train', 'images/val', 'labels/train', 'labels/val')
    missing = [part for part in required if not (DATASET_DIR / part).is_dir()]
    if missing:
        raise FileNotFoundError(
            f'No data.yaml in {DATASET_DIR}; missing folders: {", ".join(missing)}'
        )
    DATA_YAML = OUTPUT_DIR / 'dataset_config.yaml'
    DATA_YAML.write_text(yaml.safe_dump({
        'path': str(DATASET_DIR),
        'train': 'images/train',
        'val': 'images/val',
        'names': ['object'],
    }, sort_keys=False), encoding='utf-8')
    print('Created dataset config:', DATA_YAML)

os.chdir(OUTPUT_DIR)
print('Dataset:', DATASET_DIR)
print('Dataset config:', DATA_YAML)
print('Results:', OUTPUT_DIR / 'runs' / 'custom')


## 3. Train your model

Set \`LABEL\` to the object you want to detect. Use \`GRID = '2x8'\` for wide, horizontal objects such as license plates; use \`4x4\` for more balanced shapes. \`SOURCE_CLASS\` selects one class ID from the annotation files. The default 300 + 20 epochs are the full two-stage recipe; lower them only for a quick pipeline check.

Training writes each seed's \`base.pt\`, \`base_last.pt\`, \`final.pt\`, \`final_last.pt\`, \`colosseum.pt\`, and \`metrics.json\` directly to \`MyDrive/Colosseum/runs/custom/RUN_NAME/seedN/\`. Checkpoints are saved during training, so completed stages remain in Drive if a later stage is interrupted.


In [ ]:
RUN_NAME = 'my_object'
LABEL = 'my object'
GRID = '4x4'
SOURCE_CLASS = 0
SEEDS = (0,)  # Use (0, 1, 2) for a three-seed experiment.
BASE_EPOCHS = 300
SCORE_EPOCHS = 20
BATCH_SIZE = 16

from colosseum import Colosseum
model = Colosseum()
checkpoints = model.train(
    data=DATA_YAML, name=RUN_NAME, label=LABEL, grid=GRID,
    source_class=SOURCE_CLASS, seeds=SEEDS,
    base_epochs=BASE_EPOCHS, score_epochs=SCORE_EPOCHS,
    batch_size=BATCH_SIZE,
)
print('Checkpoints:', *checkpoints, sep='\n- ')


## 4. Read validation metrics

Each seed writes `metrics.json` beside its checkpoint. AP50, AP75 and AP50:95 are computed on the validation split. Precision, recall and F1 use confidence 0.35 and IoU 0.50.


In [ ]:
import json

metric_keys = ('ap50', 'ap75', 'ap50_95', 'precision', 'recall', 'f1')
for checkpoint in checkpoints:
    report = json.loads(checkpoint.with_name('metrics.json').read_text())
    print(f'Seed {report["seed"]} | ' + ' | '.join(
        f'{key}={report[key]:.4f}' for key in metric_keys
    ))
print('First seed metrics are also available as model.metrics')


## 5. Try prediction on one validation image

This is a visual check on an existing validation image. The saved checkpoint can later be loaded with `Colosseum(checkpoints[0])`.


In [ ]:
import yaml
from PIL import Image, ImageDraw, ImageOps
from IPython.display import display

config = yaml.safe_load(DATA_YAML.read_text())
base = Path(config.get('path', DATA_YAML.parent))
if not base.is_absolute():
    base = (DATA_YAML.parent / base).resolve()
val_images = base / config['val']
image_paths = sorted(p for p in val_images.rglob('*') if p.suffix.lower() in
                     {'.jpg', '.jpeg', '.png', '.bmp', '.webp'})
if not image_paths:
    raise FileNotFoundError(f'No validation images found in {val_images}')
image_path = image_paths[0]
boxes = model.predict(image_path)
with Image.open(image_path) as opened:
    preview = ImageOps.exif_transpose(opened).convert('RGB')
draw = ImageDraw.Draw(preview)
for box in boxes:
    draw.rectangle((box.x1, box.y1, box.x2, box.y2), outline='red', width=3)
    draw.text((box.x1, max(0, box.y1 - 12)),
              f'{box.label} {box.confidence:.2f}', fill='red')
print(image_path, '| detections:', len(boxes))
display(preview)


## 6. Find all saved files in Google Drive

Every \`.pt\` checkpoint and \`metrics.json\` is already in your Drive output folder. This cell lists the files for each seed.


In [ ]:
for checkpoint in checkpoints:
    seed_dir = checkpoint.parent
    print(f'\n{seed_dir}')
    for result in sorted(seed_dir.glob('*.pt')):
        print(' -', result.name, f'({result.stat().st_size / 1024 / 1024:.1f} MiB)')
    metrics_path = seed_dir / 'metrics.json'
    if metrics_path.is_file():
        print(' -', metrics_path.name)
print('Google Drive results:', OUTPUT_DIR / 'runs' / 'custom' / RUN_NAME)
